In [0]:
# XSD PARSER UTILITY
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

import xml.etree.ElementTree as ET

def extract_metadata(xsd_path):
    # Parse XSD
    tree = ET.parse(xsd_path)
    # Get root
    root = tree.getroot()
    # Namespace
    ns = {
        'xs': 'http://www.w3.org/2001/XMLSchema'
    }

    # Extract all XSD elements
    elements = root.findall(".//xs:element", ns)
    metadata = []

    for elem in elements:
        # Skip container elements
        child_elements = elem.findall(
            ".//xs:element",
            ns
        )

        if len(child_elements) > 0:
            continue

        column_name = elem.attrib.get("name")

        data_type = elem.attrib.get(
            "type",
            "xs:string"
        )

        min_occurs = elem.attrib.get(
            "minOccurs",
            "1"
        )

        max_occurs = elem.attrib.get(
            "maxOccurs",
            "1"
        )

        metadata.append({
            "column_name": column_name,
            "xsd_type": data_type,
            "min_occurs": min_occurs,
            "max_occurs": max_occurs,
            "parent": None,
            "path": column_name,
            "level": 0
        })

    schema = StructType([
        StructField("column_name", StringType(),True),
        StructField("xsd_type",StringType(),True),
        StructField("min_occurs",StringType(),True),
        StructField("max_occurs",StringType(),True),
        StructField("parent",StringType(),True),
        StructField("path",StringType(),True),
        StructField("level",IntegerType(),True)
    ])

    metadata_df = spark.createDataFrame(
        metadata,
        schema=schema
    )
    return metadata_df